# ML-04 ? Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ranahassan9/Machine-Learning-FlyRank-AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This is the contract notebook for our Week 3 lane. It is not a model-building notebook yet; it is a clear write-down of what a row means, which fields are safe to use, and which fields are forbidden.

We follow the lane from Week 1 and Week 2:
- We care about page-level search decline and AI traffic as a guardrail.
- The main prediction target is a page that is declining in traditional search.
- We treat AI traffic as a business signal to protect, not as the thing we pretend to explain.

This notebook makes the logic honest before any model or action rule is built.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


In [5]:
import pandas as pd

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# One row = one content item (page). This is a page-level snapshot, not a daily time series.
# The columns with 90-day / 30-day names are the evidence that the window is trailing and aggregated.
window_cols = [c for c in df.columns if c.endswith("_90d") or c.endswith("_30d") or c in ["ctr", "avg_position", "trend_pct", "trend_direction"]]

print(f"Rows: {len(df):,}")
print(f"Unique content IDs: {df['content_id'].nunique():,}")
print(f"Duplicate content IDs: {df['content_id'].duplicated().sum():,}")
print(f"Unique clients: {df['client_id'].nunique():,}")
print("Sample window columns:", window_cols[:12])
print("\nExample row:")
print(df[["content_id", "client_id", "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d", "trend_direction"]].head(3).to_string(index=False))


Rows: 30,000
Unique content IDs: 30,000
Duplicate content IDs: 0
Unique clients: 32
Sample window columns: ['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d']

Example row:
          content_id         client_id  impressions_90d  clicks_90d  sessions_90d  ai_sessions_90d trend_direction
content_304f48230142 client_f369cb89fc             3803          29            17                0            down
content_a1fb4e703a9e client_4e07408562            15320           7             9                0            down
content_9aa793d4d895 client_7f2253d7e2            12581          11            11                0            down


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Simple English: this is the rule book for the table.

- Feature: the things we can know before the decision is made and use to explain or predict decline.
- Label: the thing we are trying to predict.
- Context: useful IDs and grouping information, but not a signal the model should learn from.
- Excluded: fields we purposely leave out because they leak the answer or are private / future-looking.

### What we will use
- Feature examples: `word_count`, `impressions_90d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_sessions_90d`, `content_type`, `main_intent`, `impression_tier`, `position_tier`
- Label: `is_declining_label` = `trend_direction == "down"`
- Context: `content_id`, `client_id`, `provider_used`, `model_used`
- Excluded: `trend_direction`, `trend_pct` because they are the source of the label. They are not honest features.

Why this matches Week 1 and Week 2:
- We are predicting whether a page is declining in traditional search.
- We keep AI traffic as a signal to protect, but we do not hide label leakage inside the feature set.
- We do not use client or item IDs as actual model features because they are just identifiers.


In [6]:
import pandas as pd

# We load the same starter dataset and then split the columns into clear buckets.
df = pd.read_csv("https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv")

# The label in this project is the decline flag, built from the trend direction.
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

features = [
    "word_count", "impressions_90d", "clicks_90d", "sessions_90d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_sessions_90d", "ai_traffic_pct", "content_type", "main_intent",
    "impression_tier", "position_tier"
]
label_columns = ["trend_direction", "is_declining_label"]
context_columns = ["content_id", "client_id", "provider_used", "model_used"]
excluded_columns = ["trend_direction", "trend_pct", "is_declining_label"]

print("Feature count:", len(features))
print("Feature sample:", features[:8])
print("\nLabel columns:", label_columns)
print("Context columns:", context_columns)
print("Excluded columns:", excluded_columns)

# Safety check: make sure we did not accidentally include the label source inside honest features.
print("\nFeature-label overlap:", sorted(set(features) & set(label_columns)))
print("Feature-context overlap:", sorted(set(features) & set(context_columns)))


Feature count: 14
Feature sample: ['word_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']

Label columns: ['trend_direction', 'is_declining_label']
Context columns: ['content_id', 'client_id', 'provider_used', 'model_used']
Excluded columns: ['trend_direction', 'trend_pct', 'is_declining_label']

Feature-label overlap: []
Feature-context overlap: []


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

This is the most important section. We are not guessing; we are checking the data with real numbers.

We verify four things:
1. Grain: each row is a unique page, not multiple rows for the same page.
2. Counts: the dataset has the size we expect.
3. Missing values: some columns are blank by design, and we need to know which ones.
4. Windows: the column names and the numbers show we are working on a 90-day trailing window, not a daily time series.


In [7]:
import pandas as pd

# Load the same dataset used in the project.
df = pd.read_csv("https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv")

print("=== Grain check ===")
print(f"Total rows: {len(df):,}")
print(f"Unique content IDs: {df['content_id'].nunique():,}")
print(f"Duplicate content IDs: {df['content_id'].duplicated().sum():,}")

print("\n=== Count check ===")
print(f"Unique clients: {df['client_id'].nunique():,}")
print(df['content_type'].value_counts().head().to_string())

print("\n=== Missingness check ===")
missing = df[["word_count", "competition_level", "ai_sessions_90d", "trend_pct", "avg_position"]].isna().mean().sort_values(ascending=False)
print(missing.round(4).to_string())

print("\n=== Window check ===")
window_summary = df[["impressions_90d", "clicks_90d", "sessions_90d", "impressions_last_30d", "impressions_prev_30d", "trend_pct"]].describe().round(3)
print(window_summary.to_string())

print("\n=== Category pattern check ===")
if "content_type" in df.columns:
    missing_by_type = df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean()).sort_values(ascending=False)
    print(missing_by_type.head().to_string())

# Interpretation:
# 1) Duplicate content IDs = 0 means one row per page.
# 2) Unique clients = 32 matches the data dictionary.
# 3) Missing values are not random; they follow content type and measurement availability.
# 4) The presence of 90-day and 30-day columns shows the page-level trailing windows we are using.


=== Grain check ===
Total rows: 30,000
Unique content IDs: 30,000
Duplicate content IDs: 0

=== Count check ===
Unique clients: 32
content_type
keyword article       27207
feedly article         2096
comparison article      697

=== Missingness check ===
word_count           0.2566
trend_pct            0.1129
competition_level    0.0870
ai_sessions_90d      0.0000
avg_position         0.0000

=== Window check ===
       impressions_90d  clicks_90d  sessions_90d  impressions_last_30d  impressions_prev_30d  trend_pct
count        30000.000   30000.000     30000.000             30000.000             30000.000  26612.000
mean          5200.366      16.097        37.067              1429.059              1783.078     -4.786
std          16838.020      75.077       107.069              5643.852              6150.430    473.862
min              1.000       0.000         1.000                 0.000                 0.000   -100.000
25%             81.000       0.000         2.000               

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This is where we stay honest. The data is useful, but it has clear limits.

1. It is a page-level sum over a trailing window. It is not a daily event log.
2. `avg_position = 0` means "no position data" rather than "rank zero". That matters because a missing value is not a real ranking score.
3. `trend_pct` and `trend_direction` are derived from comparison windows. They are not normal feature inputs because they encode the label signal itself.
4. The dataset is not a causal experiment. It shows correlations and measured patterns, not proven cause-and-effect.
5. Some pages have no keyword context or weak search history. That means a missing value is often informative, not random noise.

This is exactly why we build a careful contract before any model: we want to avoid leaking the answer into the feature table and we want to avoid pretending the data proves more than it can prove.


In [8]:
import pandas as pd

# Data limits check: what the dataset cannot honestly tell us.
df = pd.read_csv("https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv")

no_position_data = (df["avg_position"] == 0).sum()
no_prev_window = (df["impressions_prev_30d"] == 0).sum()
trend_missing = df["trend_pct"].isna().sum()
interesting_rows = ((df["impressions_90d"] == 0) & (df["clicks_90d"] > 0)).sum()

print(f"Rows where avg_position == 0: {no_position_data:,}")
print(f"Rows where impressions_prev_30d == 0: {no_prev_window:,}")
print(f"Rows where trend_pct is missing: {trend_missing:,}")
print(f"Rows with zero impressions but positive clicks: {interesting_rows:,}")

# Interpretation:
# - If avg_position == 0 is common, then zero does not mean rank zero. It means no position data.
# - Many rows have zero previous-window impressions, which is why the trend label is treated carefully.
# - Missing trend data is a real data quality issue, so we do not pretend it is a normal feature.
# - This dataset is a measured snapshot, not a perfect causal ground truth.


Rows where avg_position == 0: 1,205
Rows where impressions_prev_30d == 0: 3,388
Rows where trend_pct is missing: 3,388
Rows with zero impressions but positive clicks: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled ? markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime ? Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` ? then submit your repo URL on the card. Done.

This notebook follows the data-contract rule from the repo: write the contract, verify it with queries, and stay honest about what the data can and cannot tell us.
